# MRT-Praktikum · Eine Schicht, zwei Echos
## Experiment 3: Einfache Dixon-Auswertung und optionale B₀-Karte
**Aufgabenfassung für Studierende.**

Sie untersuchen eine einzelne Schichtebene aus den vorhandenen Classic-DICOM-Daten.
Führen Sie zunächst die Installationszelle aus und starten Sie anschließend den Kernel neu. Beginnen Sie danach mit Schritt 1.
`=` speichert einen Wert unter einem Namen; `dicom` bezeichnet danach das geladene Bild samt Metadaten.
**Nach der Vorbereitung beginnen Sie mit dem Laden der DICOM-Datei in Schritt 1.**

**0 · Bibliotheken installieren**

Führen Sie die folgende Codezelle mit **Shift + Enter** aus und warten Sie, bis die Installation abgeschlossen ist. Die Paketliste enthält die für das Praktikum benötigten Bibliotheken.

Starten Sie anschließend über **Kernel → Restart Kernel…** den Kernel neu und fahren Sie mit **Schritt 1** fort. Wenn die Bibliotheken bereits im gewählten Kernel installiert sind, können Sie diese Vorbereitungszelle überspringen.

In [ ]:
%pip install -r requirements-jupyterhub.txt

**1 · Eine DICOM-Datei laden**

Die erste Datei ist vorgegeben: `wasser_oel_dual_echo2_magnitude.dcm` gehört zur Wasser-/Ölflasche ohne Metallknopf und enthält die Magnitude des zweiten Dual-Echos.
Anführungszeichen kennzeichnen den Dateipfad als Text. Die tatsächliche Echozeit lesen Sie anschließend aus den DICOM-Metadaten.

In [ ]:
import pydicom
dicom = pydicom.dcmread("daten/beispiel/wasser_oel_dual_echo2_magnitude.dcm")

### So arbeiten Sie
Markdownzellen erklären die Aufgabe; Codezellen führen Python aus. Führen Sie die Zellen von oben nach unten aus.
Ersetzen Sie `...` durch den gesuchten Ausdruck. Eine Fehlermeldung an einer noch offenen Aufgabe ist normal.
Markdown bearbeiten: doppelklicken, Antwort eintragen, **Shift + Enter**. Regelmäßig **Strg + S / ⌘ + S** speichern.

Alle Ansichten bleiben auf derselben Schichtebene. Die Bildmatrix ist nicht automatisch die akquirierte Auflösung.

**2 · Pixel auslesen**

`pixel_array` ist eine rechteckige Zahlentabelle (Array). `.shape` nennt Zeilen und Spalten.

In [ ]:
pixelwerte = dicom.pixel_array
print(pixelwerte.shape)

**3 · Das erste Bild**

Matplotlib zeichnet die gespeicherten Werte als Graustufen. Diese erste Ansicht ist noch nicht physikalisch skaliert.

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(pixelwerte, cmap="gray");

**4 · Tatsächliche Echozeit**

`float` macht aus dem DICOM-Wert eine Zahl. Lesen Sie `EchoTime` aus; Classic-DICOM speichert sie in Millisekunden.

In [ ]:
te_ms = float(dicom.EchoTime)
print("Echozeit [ms]:", te_ms)

**5 · Magnitude oder Phase?**

Geben Sie den Bildtyp aus. In diesen Dateien bedeutet `M` Magnitude und `P` Phase.

In [ ]:
print(dicom.ImageType)

**Kurzantwort:** Welcher Bildtyp ist geladen und welche TE gehört dazu?

*Ihre Antwort: …*

**6a · Rechnen ohne Ganzzahlüberlauf**

Wandeln Sie die Pixel in Gleitkommazahlen um. Sonst kann die spätere Subtraktion vorzeichenloser Ganzzahlen überlaufen.

In [ ]:
pixel_float = pixelwerte.astype(float)

**6b · Skalierungsfaktoren lesen**

Slope und Intercept stehen in der Datei. Die Werte sind keine Konstanten für alle Scanner.

In [ ]:
slope = float(dicom.RescaleSlope)
intercept = float(dicom.RescaleIntercept)

**6c · Skalierung anwenden**

Verwenden Sie $S=\mathrm{Pixel}\cdot\mathrm{Slope}+\mathrm{Intercept}$. Die Magnitude hat hier relative Signalwerte, keine Gewebekonzentration.

In [ ]:
signal = ...
print("Signalbereich:", signal.min(), signal.max())

**7a · Ein Pixel**

Python beginnt bei Index 0. `signal[200, 250]` bedeutet Zeile 200, Spalte 250.

In [ ]:
print("Pixel [200, 250]:", signal[200, 250])

**7b · Einen Ausschnitt wählen**

`180:280` enthält Index 180 bis 279. Ergänzen Sie den Ausschnitt für Zeilen 180–279 und Spalten 210–289.

In [ ]:
ausschnitt = signal[...]
plt.imshow(ausschnitt, cmap="gray");

## Zwei passende Echos
Die Hilfsdatei übernimmt Dateisuche, Geometrieprüfung und Anzeige. Sie rechnet noch kein Dixon-Ergebnis aus.
Die Beispielauswahl enthält **20 DICOM-Bilder einer gemeinsamen Schichtebene**. Aufnahme, Echo und Bildtyp bleiben getrennte Merkmale.

Die Dateinamen benennen Objektzustand, Aufnahme, Echonummer und Bildtyp, z. B. `wasser_oel_dual_echo1_phase.dcm` oder `wasser_oel_mit_knopf_triple_echo3_magnitude.dcm`.
`echo1`, `echo2` und `echo3` zählen die Echos nach aufsteigender TE innerhalb der jeweiligen Aufnahme. Ob ein Echo IP oder OP ist, prüfen Sie anhand der tatsächlichen TE und des Frequenzabstands.

| Aufnahme | Echos | Bildtypen | Dateien |
| --- | --- | --- | ---: |
| Wasser/Öl · Dual-Echo | 1–2 | Magnitude und Phase | 4 |
| Wasser/Öl · Triple-Echo | 1–3 | Magnitude und Phase | 6 |
| Wasser/Öl mit Knopf · Dual-Echo | 1–2 | Magnitude und Phase | 4 |
| Wasser/Öl mit Knopf · Triple-Echo | 1–3 | Magnitude und Phase | 6 |

Die [Dateiübersicht](daten/beispiel/README.md) erläutert die Benennung. `manifest.csv` ordnet die neuen Dateinamen den ursprünglichen Exportnamen zu.
Die tatsächlichen TE, Bildtypen und Geometrien werden weiterhin aus den DICOM-Dateien gelesen.

**8a · Helfer laden**

`as mh` gibt der Hilfsbibliothek einen kurzen Namen. NumPy (`np`) stellt später Array-Rechnungen bereit.

In [ ]:
import mrt_hilfen as mh
import numpy as np

**8b · Datenordner**

`daten/beispiel/` enthält die benannten Beispieldateien; `daten/eigene/` ist für Ihre Exporte vorgesehen.
Der Katalog behält nur Echobilder mit derselben Position, Orientierung, Pixelgröße und demselben Referenzrahmen wie die erste Datei.

In [ ]:
datenordner = "daten/beispiel"
katalog = mh.katalog(datenordner, referenz=dicom)

**8c · Technische Übersicht**

Suchen Sie die Dual-Echo-Aufnahme und nennen Sie die beiden Magnituden-Echozeiten. Vollständige DICOM-Header gehören nicht in die Abgabe.

In [ ]:
mh.uebersicht(katalog)

**8d · Echo-Paar wählen**

Starten Sie mit `T1W_FFE dual echo`: frühes Echo als OP, spätes als IP. Das ist die Zuordnung des Beispielprotokolls; bei eigenen Daten anhand des Messprotokolls prüfen.

In [ ]:
auswahl = mh.echo_auswahl(katalog, "T1W_FFE dual echo")

**8e · Ausgewählte Bilder übernehmen**

Nach einer Änderung im Auswahlfeld diese und alle folgenden Zellen erneut ausführen. Der Import skaliert jedes Bild genau einmal und prüft die Paarung.

In [ ]:
op, ip, meta_op, meta_ip = mh.lade_paar(auswahl)

**8f · Echozeiten nachsehen**

`meta_ip` und `meta_op` sind kleine Nachschlagewerke (Dictionaries); der Text in eckigen Klammern wählt den Eintrag.

In [ ]:
print("OP [ms]:", meta_op["echozeit_ms"])
print("IP [ms]:", meta_ip["echozeit_ms"])

**9a · IP/OP physikalisch einordnen**

Für diese Lehrabschätzung verwenden wir **angenommene** 220 Hz chemischen Frequenzabstand. Ersetzen Sie ihn durch Ihre Messung aus Experiment 1, falls vorhanden.

In [ ]:
chemie_hz = 220.0
print("Umdrehungen OP / IP:", np.array([meta_op["echozeit_ms"], meta_ip["echozeit_ms"]]) * chemie_hz / 1000)

**Kurzantwort:** Liegen die Umdrehungen ungefähr bei halben bzw. ganzen Zahlen?

*Ihre Antwort: …*

**9b · Gemeinsames Grauwertfenster**

Beide Bilder erhalten dasselbe Darstellungsfenster (0 bis gemeinsames 99. Perzentil). Die Arrays selbst werden nicht verändert oder einzeln normiert.

In [ ]:
fenster = mh.fenster(ip, op)
fig_ip = mh.bild(ip, f"IP · TE {meta_ip['echozeit_ms']:g} ms", meta_ip, fenster)

**9c · OP vergleichen**

Vergleichen Sie dieselben Bildorte. Welche Bereiche verlieren im OP-Bild Signal?

In [ ]:
fig_op = mh.bild(op, f"OP · TE {meta_op['echozeit_ms']:g} ms", meta_op, fenster)

## Einfache Dixon-Rechnung
Im idealisierten Magnitudenmodell gilt $M_{IP}=W+F$ und $M_{OP}=|W-F|$.
Zunächst berechnen wir den **größeren** Anteil $A$ und den **kleineren** Anteil $B$.
Die Formel allein sagt noch nicht, welcher davon Wasser oder Fett ist.

**10 · Größeren Modellanteil berechnen**

Ergänzen Sie $A=(M_{IP}+M_{OP})/2$. Array-Rechnungen gelten gleichzeitig für jedes Pixel.

In [ ]:
anteil_gross = ...

**11 · Kleineren Modellanteil berechnen**

Ergänzen Sie $B=(M_{IP}-M_{OP})/2$. Verwenden Sie hier keinen Betrag und kein Clipping.

In [ ]:
anteil_klein = ...

**12a · Viereransicht**

Rote Pixel markieren negative kleinere Anteile. Dort ist das vereinfachte Modell verletzt, etwa durch Rauschen, Relaxation oder ein ungeeignetes Paar.

In [ ]:
fig_dixon = mh.vierer(ip, op, anteil_gross, anteil_klein, meta_ip, meta_op)

**12b · Die Mehrdeutigkeit ausprobieren**

Ändern Sie anschließend die Werte zu Wasser = 20 und Fett = 80. Vergleichen Sie die simulierten Magnituden.

In [ ]:
wasser_test, fett_test = 80, 20
print("IP / OP:", wasser_test + fett_test, abs(wasser_test - fett_test))

**Kurzantwort:** Warum lassen sich die beiden Mischungen nicht unterscheiden? Welche Annahme brauchen wir?

*Ihre Antwort: …*

**12c · Die bekannte Öl-Wasser-Flasche**

Die Zuordnung der Wasser- und Ölbereiche beruht auf der bekannten Befüllung und der dokumentierten Lagerung des Phantoms. Bei den Beispieldaten liegt in der dargestellten Ebene der ölreiche Bereich oben und der wasserreiche Bereich unten.

Die folgende Ansicht zeigt die geladene IP-Magnitude. Die Bildhelligkeit allein erlaubt keine eindeutige Wasser-/Fett-Zuordnung. Dokumentieren Sie bei eigenen Aufnahmen die Lage beider Flüssigkeiten und passen Sie die ROIs daran an.

In [ ]:
fig_zuordnung = mh.bild(ip, "Gewählte Ebene · Zuordnung anhand der Phantomlagerung", meta_ip, fenster)

## Interaktive ROI und Signalanteil
Eine **ROI** ist ein ausgewählter Bildbereich. Wir verwenden zuerst die markierte **Wasser-ROI**, danach die **Öl-ROI** derselben Ebene.
Die bekannte Befüllung und die dokumentierte Lagerung begründen die lokale Zuordnung: Wasser-ROI **W ≥ F**, Öl-ROI **F ≥ W**.
Die Schicht ist 10 mm dick; Übergangsbereiche können Partialvolumen enthalten. Die Signalanteile sind weder validierte PDFF noch exakte Volumenanteile.
Die vorgegebenen ROI-Koordinaten gelten für die Beispieldaten. Bei eigenen Daten die Bereiche anhand des eigenen Messprotokolls neu wählen.

**13a · Signalmaske**

Wir werten nur Pixel oberhalb von 15 % des 99. IP-Perzentils aus. Diese heuristische Schwelle ersetzt keine Rauschmessung. Ergänzen Sie den Vergleich `ip > schwelle`.

In [ ]:
schwelle = 0.15 * np.percentile(ip, 99)
signalmaske = ...

**13b · Wasser-ROI mit Schiebereglern wählen**

Starten Sie unten im wasserreichen Bereich: Zeilen 350:410, Spalten 310:360. Die grüne Fläche wird live aktualisiert; Rechteck und Signalmaske werden kombiniert.

In [ ]:
roi_box = mh.roi_waehlen(ip, meta_ip, signalmaske)

**13c · Auswahl für die Rechnung übernehmen**

Diese Zelle kopiert den aktuellen ROI-Zustand. Nach jeder Regleränderung ab hier erneut ausführen; bereits berechnete Variablen ändern sich nicht rückwirkend.

In [ ]:
roi = roi_box.maske
mh.pruefe_roi(roi)

**13d · Modellverletzungen zählen**

Negative Werte bleiben in der Rechnung enthalten. Ihr Anteil zeigt, wie kritisch das Ergebnis interpretiert werden muss.

In [ ]:
negativ_prozent = 100 * np.mean(anteil_klein[roi] < 0)
print("ROI-Pixel mit B < 0 [%]:", negativ_prozent)

**14a · Wasser-ROI zuordnen**

In dieser Wasser-ROI ordnen wir unter **W ≥ F** den größeren Anteil dem Wasser zu. Die Zuordnung ist durch das Phantomwissen begründet und gilt nicht automatisch im ölreichen Bereich.

In [ ]:
wasser = anteil_gross
fett = anteil_klein

**14b · ROI-Mittelwerte**

`array[roi]` wählt alle gültigen Pixel. `.mean()` bildet deren Mittelwert. Ergänzen Sie beide Mittelwerte.

In [ ]:
wasser_mittel = ...
fett_mittel = ...

**15a · Bedingten Fett-Signalanteil berechnen**

Berechnen Sie $100\,\overline{F}/(\overline{W}+\overline{F})$. Das ist ein **Quotient von Mittelwerten**, kein Mittelwert pixelweiser Quotienten.

In [ ]:
fett_signal_prozent = ...
print("Fett-Signalanteil unter W ≥ F [%]:", fett_signal_prozent)

**Kurzantwort:** Was ändert sich bei fettdominanter Zuordnung? Wie belastbar ist der Prozentwert?

*Ihre Antwort: …*

**Kurzantwort:** Warum ist ein kleinerer Anteil im Übergangsbereich nicht automatisch der Fettanteil?

*Ihre Antwort: …*

**15b · Ergebnisabbildung sichern**

Die Dateien landen im Ordner `ergebnisse`. Der ROI-Export zeigt die zuletzt gewählte grüne Fläche.

In [ ]:
mh.speichere_bild(fig_dixon, "dixon.png")
mh.speichere_bild(roi_box.figure, "dixon_roi.png")

**15c · ROI-Tabelle sichern**

Die Tabelle enthält Datei, TE, Position, ROI-Grenzen, gültige Fläche und Ergebnis mit der ausdrücklich benannten Zuordnungsannahme.

In [ ]:
mh.protokoll("dixon_roi.csv", meta_op, meta_ip, roi_box,
             fett_unter_W_ge_F_prozent=fett_signal_prozent, negativ_prozent=negativ_prozent)

**15d · Jetzt die Öl-ROI wählen**

Wählen Sie oben den ölreichen Bereich: Zeilen 100:160, Spalten 120:170. Die Wasser-ROI bleibt separat gespeichert.

In [ ]:
oel_box = mh.roi_waehlen(ip, meta_ip, signalmaske, start=(100, 160, 120, 170))

**15e · Öl-Auswahl übernehmen**

Nach Änderungen an diesem Regler ab hier erneut ausführen.

In [ ]:
roi_oel = oel_box.maske
mh.pruefe_roi(roi_oel)

**15f · Die Zuordnung umkehren**

In der Öl-ROI gilt **F ≥ W**: Der größere Modellanteil gehört zum Fett-/Ölsignal, der kleinere zum Wassersignal. Ergänzen Sie die Mittelwerte in dieser ROI.

In [ ]:
fett_oel_mittel = ...
wasser_oel_mittel = ...

**15g · Öl-ROI auswerten**

Berechnen Sie wieder den Quotienten der Mittelwerte. Vergleichen Sie das Ergebnis mit der Wasser-ROI.

In [ ]:
fett_oel_prozent = ...
print("Fett-/Öl-Signalanteil unter F ≥ W [%]:", fett_oel_prozent)

**15h · Modellverletzungen in der Öl-ROI**

Auch hier negative kleinere Anteile weder abschneiden noch durch Beträge verstecken.

In [ ]:
negativ_oel = 100 * np.mean(anteil_klein[roi_oel] < 0)
print("Öl-ROI: negative kleinere Anteile [%]:", negativ_oel)

**15i · Öl-Ergebnis sichern**

Das zweite Ergebnis erhält einen eigenen Dateinamen und die andere Zuordnungsannahme.

In [ ]:
mh.protokoll("dixon_oel_roi.csv", meta_op, meta_ip, oel_box,
             fett_unter_F_ge_W_prozent=fett_oel_prozent, negativ_prozent=negativ_oel)

**15j · Beide Bereiche dokumentieren**

Speichern Sie auch die zweite markierte ROI für die Abgabe.

In [ ]:
mh.speichere_bild(oel_box.figure, "dixon_oel_roi.png")

**Kurzantwort:** Was zeigt der Vergleich der bekannten Wasser- und Ölbereiche?

*Ihre Antwort: …*

## Eigene Daten und kurze Diskussion
1. Legen Sie die eigenen **Classic-DICOM-Echobilder** unter `daten/eigene/` ab (auch Dateinamen ohne `.dcm` funktionieren).
2. Ändern Sie den Dateipfad in **Schritt 1** auf eine Magnitudendatei der gewünschten Ebene. Setzen Sie in **8b** `datenordner = "daten/eigene"`.
3. Wählen Sie in **8d** Ihre Aufnahme und die anhand des Protokolls passenden OP-/IP-Echos. Führen Sie ab Schritt 1 erneut aus. Die Referenzdatei legt genau eine Ebene fest.
4. Passen Sie ROI und gegebenenfalls Signalschwelle an. Echozeiten und Skalierungen werden neu aus den Dateien gelesen.

Sie können für eigene Exporte die ursprünglichen Dateinamen beibehalten. Der Import erkennt Aufnahme, Bildtyp und Echozeit über DICOM-Metadaten.

Keine eigenen Daten vorhanden? Nutzen Sie `T1W_FFE dual echo button` auf derselben Ebene als zweite Aufnahme. Die zugehörigen Dateien beginnen mit `wasser_oel_mit_knopf_dual_`.
Beim Wechsel zu einer anderen Untersuchung müssen auch Referenzdatei und Katalog neu geladen werden.

**Kontrollpunkt:** 2D-Bildpaar, tatsächliche TE, Viereransicht, Wasser- und Öl-ROI, lokale Zuordnung und kurze Antworten liegen vor.
Sie können jetzt zur Abgabe am Notebookende springen. B₀ ist optional.

## Optional: Näherungsweise B₀-Karte
Wir verwenden **direkt exportierte Phasenbilder** der äußeren Triple-Echos, keine Real-/Imaginär-Rekonstruktion.
Diese Echos sind näherungsweise chemisch gleichphasig. Im Beispiel liegen drei chemische Umdrehungen zwischen ihnen;
ein exakt realisiertes 2π-/4π-Paar wird nicht vorausgesetzt.

Die Interpretation setzt konstante Empfangsphase, Objektlage, Shim und Frequenzeinstellung voraus.
Verwenden Sie für den Einstieg die signalstarke Wasser-ROI unten in der Flasche, nicht die Übergangszone.
Auch im wasserreichen Bereich bleibt die Karte eine Näherung; die gesamte Schicht wird nicht als reines Wasser behandelt.

**B1 · Phasenpaar wählen**

Die Voreinstellung verwendet die äußeren Triple-Echos. Ein beliebiges IP-/OP-Phasenpaar ist für diese einfache B₀-Aufgabe ungeeignet.

In [ ]:
phase_auswahl = mh.echo_auswahl(katalog, "T1W_FFE triple echo", modus="b0")

**B2 · Phasen laden**

Die Hilfsfunktion liest die Skalierung und Einheit aus DICOM, rechnet mrad in rad um und ordnet nach tatsächlicher TE.

In [ ]:
phase_1, phase_2, meta_1, meta_2 = mh.lade_paar(phase_auswahl)

**B3 · Tatsächliche TE übernehmen**

Lesen Sie die beiden Echozeiten selbst aus den Metadaten. Im Auswertecode werden keine Beispiel-Echozeiten eingetragen.

In [ ]:
te_1_ms = meta_1["echozeit_ms"]
te_2_ms = meta_2["echozeit_ms"]

**B4 · TE kontrollieren**

Prüfen Sie Reihenfolge und Einheit vor der Division.

In [ ]:
print("Echo 1 [ms]:", te_1_ms)
print("Echo 2 [ms]:", te_2_ms)

**B5 · Zeitabstand berechnen**

Subtrahieren Sie die tatsächlichen Echozeiten und rechnen Sie Millisekunden in Sekunden um.

In [ ]:
delta_te_ms = ...
delta_te_s = ...

**B6 · Chemische Näherung prüfen**

Die Umdrehungszahl sollte ungefähr ganzzahlig sein. Die 220-Hz-Annahme bleibt eine Näherung und kein Messnachweis.

In [ ]:
print("ΔTE [s]:", delta_te_s)
print("Chemische Umdrehungen im Abstand:", chemie_hz * delta_te_s)

**B7 · Erste Phase anzeigen**

Die Originalskalierung des Exports wird verwendet. Eine willkürliche Normierung auf −π bis π wäre falsch.

In [ ]:
fig_phase_1 = mh.bild(phase_1, f"Phase · TE {te_1_ms:g} ms [rad]", meta_1, (-np.pi, np.pi), "twilight")

**B8 · Zweite Phase anzeigen**

Dieselbe Farbskala erlaubt den Vergleich. Die Außenbereiche sind verrauscht und werden für die Rechnung maskiert.

In [ ]:
fig_phase_2 = mh.bild(phase_2, f"Phase · TE {te_2_ms:g} ms [rad]", meta_2, (-np.pi, np.pi), "twilight")

**B9 · Zugehörige Magnituden laden**

Für die Signalmaske benötigen wir Magnituden genau derselben Aufnahme, Ebene und Echozeiten wie die Phasen.

In [ ]:
betrag_1 = mh.magnitude_zu_phase(katalog, meta_1)
betrag_2 = mh.magnitude_zu_phase(katalog, meta_2)

**B10 · Signal in beiden Echos verlangen**

Der Schwellwert wird für jedes Echo aus dessen 99. Perzentil bestimmt. `&` bedeutet: beide Bedingungen müssen stimmen. Erhöhen Sie bei Bedarf den Faktor.

In [ ]:
schwellen_b0 = 0.15 * np.array([np.percentile(betrag_1, 99), np.percentile(betrag_2, 99)])
signal_b0 = (betrag_1 > schwellen_b0[0]) & (betrag_2 > schwellen_b0[1])

**B11 · Zusammenhängende innere ROI wählen**

Die grüne Fläche muss innerhalb eines signalstarken Objektbereichs liegen. Getrennte Teile können verschiedene unbekannte Phasenoffsets behalten.

In [ ]:
b0_box = mh.roi_waehlen(betrag_1, meta_1, signal_b0)

**B12 · Maske übernehmen und prüfen**

Nach Regleränderungen ab dieser Zelle erneut ausführen. Bei mehreren getrennten Bereichen ein kleineres Rechteck im Objektinneren wählen.

In [ ]:
maske_b0 = b0_box.maske
mh.pruefe_roi(maske_b0, zusammenhaengend=True)

**B13 · Zirkuläre Phasendifferenz**

Die vorbereitete Zeile berechnet $\arg\{\exp[i(\phi_2-\phi_1)]\}$. Sie entfernt Darstellungs-Sprünge einzelner Phasen, ist aber noch kein räumliches Unwrapping.

In [ ]:
delta_phase = np.angle(np.exp(1j * (phase_2 - phase_1)))

**B14 · Maskiert entfalten**

Die bereitgestellte Funktion arbeitet nur innerhalb der geprüften ROI. Außerhalb steht NaN (kein gültiger Wert). Unwrapping bestimmt keinen absoluten globalen Offset.

In [ ]:
phase_entfaltet = mh.entfalte(delta_phase, maske_b0)
fig_entfaltet = mh.phasenvergleich(delta_phase, phase_entfaltet, maske_b0, meta_1)

**B15 · Von rad nach Hz**

Ergänzen Sie $\Delta f=\Delta\phi/(2\pi\,\Delta TE)$. Verwenden Sie den **gesamten** Echoabstand in Sekunden; kein zusätzlicher Faktor 1/2.

In [ ]:
feld_hz = ...

**B16 · Referenzfrequenz lesen**

Die DICOM-Frequenz steht in MHz. Rechnen Sie in Hz um; die Datei liefert die tatsächliche Frequenz.

In [ ]:
referenz_hz = ...
print("Referenzfrequenz [Hz]:", referenz_hz)

**B17 · Von Hz nach ppm**

Ergänzen Sie $\Delta B_0[\mathrm{ppm}]=10^6\,\Delta f/f_0$. Das Vorzeichen folgt hier der Konvention $\phi=\phi_0+2\pi\Delta f\,TE$; eine unabhängige Scanner-Vorzeichenkalibrierung liegt nicht vor.

In [ ]:
feld_ppm = ...

**B18 · Nicht zentrierte Karte ansehen**

Titel und Maske machen die eingeschränkte Aussage sichtbar. Die Farbskala ist ±0,6 ppm; ändern Sie sie bei Sättigung für beide Karten gemeinsam.

In [ ]:
farbgrenze = 0.6
fig_b0 = mh.feldkarte(feld_ppm, maske_b0, meta_1, "Näherungsweise B₀ [ppm] · globaler Offset offen", farbgrenze)

**B19 · Relative Karte**

Ziehen Sie den ROI-Mittelwert ab. Ein gemeinsamer konstanter Offset spielt dann für die Homogenitätsmaße innerhalb dieser ROI keine Rolle.

In [ ]:
offset_ppm = feld_ppm[maske_b0].mean()
feld_relativ = ...

**B20 · Relative Homogenität darstellen**

Dies ist nur die ausgewählte ROI einer Schicht, keine Volumen-Homogenität. Ausgeblendete Pixel sind fehlende Information.

In [ ]:
fig_b0_rel = mh.feldkarte(feld_relativ, maske_b0, meta_1, "Relative B₀ [ppm] · Mittelwert abgezogen", farbgrenze)

**B21 · Streuungsmaße**

Berechnen Sie Standardabweichung und Maximum minus Minimum. Peak-to-Peak reagiert empfindlich auf einzelne Ausreißer.

In [ ]:
std_ppm = ...
spanne_ppm = ...

**B22 · Kennwerte ausgeben**

Berichten Sie auch den nicht absolut kalibrierten Mittelwert. Eine kleine Streuung allein beweist keine korrekte Feldrekonstruktion.

In [ ]:
print("ROI-Mittel / Standardabweichung [ppm]:", offset_ppm, std_ppm)
print("Peak-to-Peak [ppm]:", spanne_ppm)

**B23 · Eindeutigkeitsgrenze**

Ohne zusätzliche Referenz gilt modulo $1/\Delta TE$. Das um null zentrierte Intervall liegt zwischen $-1/(2\Delta TE)$ und $+1/(2\Delta TE)$ (obere Grenze ausgeschlossen).

In [ ]:
grenze_hz = 1 / (2 * delta_te_s)
print("Nicht entfaltetes Intervall: ±", grenze_hz, "Hz; ±", 1e6 * grenze_hz / referenz_hz, "ppm")

**Kurzantwort:** Welche Aussage erlaubt die B₀-Karte, und welche nicht?

*Ihre Antwort: …*

**B24 · Abbildungen sichern**

Beide Karten verwenden dieselbe Farbgrenze. Speichern Sie zusätzlich die ROI-Ansicht.

In [ ]:
mh.speichere_bild(fig_b0, "b0_nicht_zentriert.png")
mh.speichere_bild(fig_b0_rel, "b0_relativ.png")

**B25 · Maske sichern**

Die ROI-Ansicht zeigt die ausgewertete Fläche innerhalb der ganzen Schicht.

In [ ]:
mh.speichere_bild(b0_box.figure, "b0_roi.png")

**B26 · Tabelle sichern**

Die Tabelle dokumentiert die tatsächlichen Echos, ROI und maskiertes 2D-Unwrapping. Ein Funktionsaufruf darf innerhalb seiner Klammern über zwei Zeilen gehen.

In [ ]:
mh.protokoll("b0_roi.csv", meta_1, meta_2, b0_box, delta_te_s=delta_te_s, offset_ppm=offset_ppm,
             std_ppm=std_ppm, peak_to_peak_ppm=spanne_ppm, unwrapping="maskiert_2D")

## Abschluss und Abgabe
- Speichern Sie das bearbeitete Notebook als eigene Datei. Prüfen Sie, dass alle bearbeiteten Zellen ohne Fehler laufen und die Abbildungen sichtbar sind.
- Pflicht: Aufnahme und tatsächliche TE, Viereransicht, Wasser- und Öl-ROI mit Grenzen, beide Signalanteile **mit lokaler Zuordnung**, Anteile negativer Pixel sowie Ihre Kurzantworten.
- Optional B₀: tatsächliches ΔTE, Referenzfrequenz, Maske, nicht zentrierte und relative ppm-Karte, Unwrapping-Angabe und ROI-Kennwerte.
- Geben Sie das Notebook zusammen mit den relevanten PNG-/CSV-Dateien aus `ergebnisse/` ab. Widgets sind in einer statischen Ansicht nicht bedienbar; PNGs und CSVs halten die letzte Auswahl fest.

**Ergebnis in zwei Sätzen:** …

**Grenze der Interpretation:** …

Zur Vertiefung: Vergleichen Sie `… button` auf **derselben Ebene**; prüfen Sie erneut Signalmaske und beide ROIs. Verwenden Sie für beide Aufnahmen dieselben Darstellungsgrenzen. Die bekannte Befüllung ist Öl/Wasser; genaue Befüllungsmengen und lokale Partialvolumenanteile sind nicht dokumentiert.

### Quellen und methodischer Bezug
Konzept: `../idee.md`; lokale Versuchsskripte und bereitgestellte September-DICOM-Daten.

- [pydicom: Rescale](https://pydicom.github.io/pydicom/stable/reference/generated/pydicom.pixels.apply_rescale.html) — Skalierung gespeicherter Pixelwerte.
- [SDCFlows: Phasendifferenz und Feldkarte](https://www.nipreps.org/sdcflows/2.12/methods.html) — Zusammenhang zwischen Phase, Echoabstand und Hz.
- [scikit-image: unwrap_phase](https://scikit-image.org/docs/stable/api/skimage.restoration.html#skimage.restoration.unwrap_phase) — maskiertes räumliches Unwrapping.
- [Dixon 1984](https://doi.org/10.1148/radiology.153.1.6089263) — ursprüngliche Methode; hier wird bewusst die vereinfachte Magnitudenübung verwendet.